In [1]:
# 01 패키지
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
# 02 랜덤 고정값
np.random.seed(42)
random.seed(42)

In [3]:
# 03 경로 설정
DATA_DIR = Path("../data")
OUTPUT_PATH = DATA_DIR / "12.5_profile_funnel.csv"

In [6]:
# 04 profile_metrics 데이터 불러오기
profile_metrics = pd.read_csv(DATA_DIR / "12_profile_metrics.csv", encoding="utf-8-sig")

# 컬럼명 앞뒤 공백 제거
profile_metrics.columns = profile_metrics.columns.str.strip()

# 빈 문자열을 NaN으로 변환
profile_metrics = profile_metrics.replace(r"^\s*$", np.nan, regex=True)

# 전체가 빈 행인 경우 제거
profile_metrics = profile_metrics.dropna(how="all")

# influencer_id가 없는 행 제거
profile_metrics = profile_metrics.dropna(subset=["profile_metric_id"])

# 날짜 변환
profile_metrics["profile_metric_at"] = pd.to_datetime(
    profile_metrics["profile_metric_at"],
    errors="coerce"
)

# 인덱스 재정렬
profile_metrics = profile_metrics.reset_index(drop=True)

# 데이터 확인
print(profile_metrics.shape)
profile_metrics.head()

(32, 11)


,profile_metric_id,client_id,profile_metric_at,profile_followers_count,profile_new_followers_count,profile_unfollowers_count,profile_reach_count,profile_impressions_count,profile_visit_count,profile_website_click_count,profile_website_click_rate
0,promet-0001,cli-0001,2026-05-14,574,7,4,3592,4870,43,1,2.33
1,promet-0002,cli-0001,2026-05-15,577,7,4,3671,4929,93,3,3.23
2,promet-0003,cli-0001,2026-05-16,581,7,3,2933,4052,37,2,5.41
3,promet-0004,cli-0001,2026-05-17,582,5,4,5343,6534,118,9,7.63
4,promet-0005,cli-0001,2026-05-18,581,2,3,1767,3167,16,1,6.25


In [7]:
# 05 퍼널 데이터 생성
funnel_rows = []
funnel_id = 1

for _, row in profile_metrics.iterrows():
    client_id = row["client_id"]
    profile_metric_id = row["profile_metric_id"]
    funnel_at = row["profile_metric_at"]

    funnel_steps = [
        (1, "1. 노출", row["profile_impressions_count"]),
        (2, "2. 도달", row["profile_reach_count"]),
        (3, "3. 프로필 방문", row["profile_visit_count"]),
        (4, "4. 웹사이트 클릭", row["profile_website_click_count"]),
    ]

    for step_no, step_name, funnel_count in funnel_steps:
        funnel_count = int(funnel_count)

        funnel_rows.append({
            "profile_funnel_id": f"profun-{funnel_id:04d}",
            "client_id": client_id,
            "profile_metric_id": profile_metric_id,
            "funnel_at": funnel_at.strftime("%Y-%m-%d %H:%M:%S"),
            "step_no": step_no,
            "step_name": step_name,
            "funnel_count": funnel_count,
            "opposite": -funnel_count
        })

        funnel_id += 1

profile_funnel = pd.DataFrame(funnel_rows)

In [8]:
# 06 CSV 저장
profile_funnel.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: ..\data\12.5_profile_funnel.csv
